# 과거 위험 후보 생성 v2.1

이 노트북은 A축 지표인 `silver_epss`와 `silver_kev`만 사용해 탐지룰을 적용하고, 과거 위험 후보 형태로 DS 스키마의 Gold 테이블에 적재합니다.

적용 기간: 2025-04-20 ~ 2026-08-19 (원천 데이터: 2025-03-21 ~)

적용 범위:

- `R-EPSS-RISE-D30`
- `R-EPSS-TOP5-ENTRY-D30`
- `R-KEV-NEW`
- 중복 처리 제외

*GHSA 심각도와 패치 관련 B/C축 룰은 이 기간의 관측 이력이 없으므로 적용하지 않습니다.

In [10]:
spark.sql(f'''
UPDATE `Watchback-workspace`.watchback_lakehouse_dev_v2.DS.gold_rereview_candidate
SET ghsa_id = 'N/A'
WHERE ghsa_id IS NULL
  AND primary_trigger_type = 'KEV_NEW';
  ''')

StatementMeta(, f7c5b1d2-9a76-44e0-9487-233a121571c6, 12, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint]

In [13]:
display(spark.sql(f'''
SELECT 
    candidate_id, 
    cve_id, 
    ghsa_id, 
    primary_trigger_type 
FROM `Watchback-workspace`.watchback_lakehouse_dev_v2.DS.gold_rereview_candidate
WHERE ghsa_id is null
LIMIT 10;
'''))


StatementMeta(, f7c5b1d2-9a76-44e0-9487-233a121571c6, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8686f3b0-ccc8-41c6-95db-31e2dc84a923)

In [1]:
from datetime import datetime, timezone
import json

LAKEHOUSE_SCHEMA = "`Watchback-workspace`.watchback_lakehouse_dev_v2"
SILVER_SCHEMA = f"{LAKEHOUSE_SCHEMA}.silver"
DS_SCHEMA = f"{LAKEHOUSE_SCHEMA}.DS"

EPSS_TABLE = f"{SILVER_SCHEMA}.silver_epss"
KEV_TABLE = f"{SILVER_SCHEMA}.silver_kev"

DS_BATCH_TABLE = f"{DS_SCHEMA}.gold_rereview_batch"
DS_CANDIDATE_TABLE = f"{DS_SCHEMA}.gold_rereview_candidate"
DS_EVIDENCE_TABLE = f"{DS_SCHEMA}.gold_rereview_candidate_evidence"


START_SNAPSHOT_DATE = "2025-04-20"
END_SNAPSHOT_DATE = "2026-08-19"

WRITE_OUTPUT_TABLES = True
POLICY_VERSION = "ver.1-historical"
SCHEMA_VERSION = "v0.1"

run_started_at = datetime.now(timezone.utc)
RUN_ID = "historical-" + run_started_at.strftime("%Y%m%dT%H%M%SZ")
BATCH_ID = "historical-" + run_started_at.strftime("%Y%m%dT%H%M%SZ")
GENERATED_AT_UTC = run_started_at.strftime("%Y-%m-%d %H:%M:%S")

print({
    "run_id": RUN_ID,
    "batch_id": BATCH_ID,
    "start_snapshot_date": START_SNAPSHOT_DATE,
    "end_snapshot_date": END_SNAPSHOT_DATE,
    "write_output_tables": WRITE_OUTPUT_TABLES,
    "input_tables": [EPSS_TABLE, KEV_TABLE],
    "output_tables": [DS_BATCH_TABLE, DS_CANDIDATE_TABLE, DS_EVIDENCE_TABLE],
})

VALIDATION_SIGNAL_VIEW = "`Watchback-workspace`.watchback_lakehouse_dev_v2.DS.historical_signal_validation_v"
VALIDATION_RULE_HITS_VIEW = "`Watchback-workspace`.watchback_lakehouse_dev_v2.DS.historical_rule_hits_validation_v"
VALIDATION_CANDIDATES_VIEW = "`Watchback-workspace`.watchback_lakehouse_dev_v2.DS.historical_candidates_validation_v"
VALIDATION_EVIDENCE_VIEW = "`Watchback-workspace`.watchback_lakehouse_dev_v2.DS.historical_candidate_evidence_validation_v"
VALIDATION_FIRST_RULE_HITS_VIEW = "`Watchback-workspace`.watchback_lakehouse_dev_v2.DS.historical_first_rule_hits_validation_v"

StatementMeta(, f7c5b1d2-9a76-44e0-9487-233a121571c6, 3, Finished, Available, Finished, False)

{'run_id': 'historical-20260828T021452Z', 'batch_id': 'historical-20260828T021452Z', 'start_snapshot_date': '2025-04-20', 'end_snapshot_date': '2026-08-19', 'write_output_tables': True, 'input_tables': ['`Watchback-workspace`.watchback_lakehouse_dev_v2.silver.silver_epss', '`Watchback-workspace`.watchback_lakehouse_dev_v2.silver.silver_kev'], 'output_tables': ['`Watchback-workspace`.watchback_lakehouse_dev_v2.DS.gold_rereview_batch', '`Watchback-workspace`.watchback_lakehouse_dev_v2.DS.gold_rereview_candidate', '`Watchback-workspace`.watchback_lakehouse_dev_v2.DS.gold_rereview_candidate_evidence']}


## 1. 입력 데이터 범위 확인

EPSS와 KEV의 날짜 범위를 확인합니다.

In [6]:
display(spark.sql(f"""
SELECT
    MIN(snapshot_date) AS min_snapshot_date,
    MAX(snapshot_date) AS max_snapshot_date,
    COUNT(DISTINCT snapshot_date) AS snapshot_date_count,
    COUNT(*) AS row_count,
    COUNT(DISTINCT cve_id) AS cve_count
FROM {EPSS_TABLE}
"""))

display(spark.sql(f"""
SELECT
    MIN(date_added) AS min_date_added,
    MAX(date_added) AS max_date_added,
    COUNT(*) AS row_count,
    COUNT(DISTINCT cve_id) AS cve_count
FROM {KEV_TABLE}
"""))

StatementMeta(, be2357d1-6e2d-494d-9071-b0380964338b, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dc2e1b1c-7dd4-446c-8b15-e6e0c1c27f86)

SynapseWidget(Synapse.DataFrame, dd2136bc-86c3-4c1c-b03d-caee9aaeae17)

## 2. 과거 후보 입력 뷰 생성

EPSS는 current 날짜와 D-30 baseline을 CVE 단위로 연결합니다.
- D-30 baseline이 있으면 EPSS_RISE, EPSS_TOP5_ENTRY를 기존 비교 조건으로 평가합니다.
- D-30 baseline이 없더라도 신규 CVE(NEW_CVE)가 현재 EPSS 위험 순위 상위 5%에 있으면 EPSS_TOP5_ENTRY로 평가할 수 있게 보정합니다.
- KEV는 date_added가 해당 날짜인 경우 신규 등재로 판단합니다.

In [7]:
spark.sql(f"""
CREATE OR REPLACE VIEW {VALIDATION_SIGNAL_VIEW} AS
WITH epss_first_seen AS (
    SELECT
        cve_id,
        MIN(CAST(snapshot_date AS date)) AS epss_first_snapshot_date
    FROM {EPSS_TABLE}
    GROUP BY cve_id
),
epss_current AS (
    SELECT
        CAST(cur.snapshot_date AS date) AS signal_snapshot_date,
        date_sub(CAST(cur.snapshot_date AS date), 30) AS lookback_snapshot_date,
        cur.cve_id,
        first_seen.epss_first_snapshot_date,
        CAST(base.snapshot_date AS date) AS epss_baseline_snapshot_date,
        CAST(base.epss_score AS double) AS epss_probability_lookback,
        CAST(cur.epss_score AS double) AS epss_current,
        CAST(base.epss_percentile AS double) AS epss_percentile_lookback,
        CAST(cur.epss_percentile AS double) AS epss_percentile_current,
        CASE
            WHEN base.cve_id IS NOT NULL THEN true
            ELSE false
        END AS has_epss_lookback_baseline,
        CASE
            WHEN base.cve_id IS NOT NULL THEN NULL
            WHEN first_seen.epss_first_snapshot_date > date_sub(CAST(cur.snapshot_date AS date), 30) THEN 'NEW_CVE'
            ELSE 'MISSING_BASELINE'
        END AS epss_lookback_missing_reason,
        CASE
            WHEN base.cve_id IS NOT NULL THEN CAST(cur.epss_score AS double) - CAST(base.epss_score AS double)
            ELSE NULL
        END AS epss_delta_d30,
        CASE
            WHEN base.cve_id IS NOT NULL AND CAST(base.epss_score AS double) > 0 THEN CAST(cur.epss_score AS double) / CAST(base.epss_score AS double)
            ELSE NULL
        END AS epss_ratio_d30
    FROM {EPSS_TABLE} cur
    LEFT JOIN {EPSS_TABLE} base
        ON cur.cve_id = base.cve_id
       AND CAST(base.snapshot_date AS date) = date_sub(CAST(cur.snapshot_date AS date), 30)
    JOIN epss_first_seen first_seen
        ON cur.cve_id = first_seen.cve_id
    WHERE CAST(cur.snapshot_date AS date) BETWEEN DATE('{START_SNAPSHOT_DATE}') AND DATE('{END_SNAPSHOT_DATE}')
),
kev_new AS (
    SELECT
        CAST(date_added AS date) AS signal_snapshot_date,
        cve_id,
        true AS kev_new_in_window
    FROM {KEV_TABLE}
    WHERE CAST(date_added AS date) BETWEEN DATE('{START_SNAPSHOT_DATE}') AND DATE('{END_SNAPSHOT_DATE}')
      AND cve_id IS NOT NULL
)
SELECT
    signal_snapshot_date,
    lookback_snapshot_date,
    cve_id,
    epss_first_snapshot_date,
    has_epss_lookback_baseline,
    epss_lookback_missing_reason,
    epss_probability_lookback,
    epss_current,
    epss_percentile_lookback,
    epss_percentile_current,
    epss_delta_d30,
    epss_ratio_d30,
    false AS kev_new_in_window
FROM epss_current

UNION ALL

SELECT
    signal_snapshot_date,
    CAST(NULL AS date) AS lookback_snapshot_date,
    cve_id,
    CAST(NULL AS date) AS epss_first_snapshot_date,
    CAST(NULL AS boolean) AS has_epss_lookback_baseline,
    CAST(NULL AS string) AS epss_lookback_missing_reason,
    CAST(NULL AS double) AS epss_probability_lookback,
    CAST(NULL AS double) AS epss_current,
    CAST(NULL AS double) AS epss_percentile_lookback,
    CAST(NULL AS double) AS epss_percentile_current,
    CAST(NULL AS double) AS epss_delta_d30,
    CAST(NULL AS double) AS epss_ratio_d30,
    kev_new_in_window
FROM kev_new
""")

display(spark.sql(f"""
SELECT
    MIN(signal_snapshot_date) AS min_signal_snapshot_date,
    MAX(signal_snapshot_date) AS max_signal_snapshot_date,
    COUNT(DISTINCT signal_snapshot_date) AS signal_snapshot_date_count,
    COUNT(*) AS row_count,
    COUNT(DISTINCT cve_id) AS cve_count,
    SUM(CASE WHEN epss_lookback_missing_reason = 'NEW_CVE' THEN 1 ELSE 0 END) AS new_cve_without_d30_count,
    SUM(CASE WHEN epss_lookback_missing_reason = 'MISSING_BASELINE' THEN 1 ELSE 0 END) AS missing_baseline_count
FROM {VALIDATION_SIGNAL_VIEW}
"""))


StatementMeta(, be2357d1-6e2d-494d-9071-b0380964338b, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 91157332-db72-41f1-bde2-64963985335b)

## 3. 과거 후보 룰 hit 생성

과거 후보 입력 뷰에서 EPSS/KEV A축 룰을 적용합니다. EPSS_RISE는 D-30 baseline이 있는 경우만 평가하고, EPSS_TOP5_ENTRY는 D-30 진입 조건과 신규 CVE 고위험 보정 조건을 함께 반영합니다.

In [8]:
spark.sql(f"""
CREATE OR REPLACE VIEW {VALIDATION_RULE_HITS_VIEW} AS
SELECT
    signal_snapshot_date,
    lookback_snapshot_date,
    cve_id,
    'R-EPSS-RISE-D30' AS rule_id,
    'EPSS_RISE' AS trigger_type,
    'epss_score' AS signal,
    45 AS score,
    CAST(epss_probability_lookback AS string) AS previous_value,
    CAST(epss_current AS string) AS current_value,
    'decimal' AS value_type,
    CAST(signal_snapshot_date AS timestamp) AS evidence_observed_at,
    'FIRST EPSS' AS source,
    CAST(signal_snapshot_date AS timestamp) AS source_snapshot_at,
    signal_snapshot_date AS source_snapshot_date,
    concat(
        'EPSS가 30일 전 대비 ',
        CAST(round(epss_ratio_d30, 2) AS string),
        '배 상승했습니다. (',
        CAST(round(epss_probability_lookback, 4) AS string),
        ' -> ',
        CAST(round(epss_current, 4) AS string),
        ')'
    ) AS evidence_summary
FROM {VALIDATION_SIGNAL_VIEW}
WHERE has_epss_lookback_baseline
  AND epss_ratio_d30 >= 5
  AND epss_delta_d30 >= 0.05

UNION ALL

SELECT
    signal_snapshot_date,
    lookback_snapshot_date,
    cve_id,
    'R-EPSS-TOP5-ENTRY-D30' AS rule_id,
    'EPSS_TOP5_ENTRY' AS trigger_type,
    'epss_percentile' AS signal,
    30 AS score,
    CAST(epss_percentile_lookback AS string) AS previous_value,
    CAST(epss_percentile_current AS string) AS current_value,
    'decimal' AS value_type,
    CAST(signal_snapshot_date AS timestamp) AS evidence_observed_at,
    'FIRST EPSS' AS source,
    CAST(signal_snapshot_date AS timestamp) AS source_snapshot_at,
    signal_snapshot_date AS source_snapshot_date,
    CASE
        WHEN epss_lookback_missing_reason = 'NEW_CVE' THEN concat(
            '신규 CVE가 EPSS 위험 순위 상위 5% 구간에 진입했습니다. (최초 관측일 ',
            date_format(epss_first_snapshot_date, 'yyyy-MM-dd'),
            ', 현재 ',
            CAST(round(epss_percentile_current, 4) AS string),
            ')'
        )
        ELSE concat(
            'EPSS 위험 순위가 30일 전 대비 상위 5% 구간에 진입했습니다. (',
            CAST(round(epss_percentile_lookback, 4) AS string),
            ' -> ',
            CAST(round(epss_percentile_current, 4) AS string),
            ')'
        )
    END AS evidence_summary
FROM {VALIDATION_SIGNAL_VIEW}
WHERE epss_percentile_current >= 0.95
  AND (
      (has_epss_lookback_baseline AND epss_percentile_lookback < 0.90)
      OR epss_lookback_missing_reason = 'NEW_CVE'
  )

UNION ALL

SELECT
    signal_snapshot_date,
    lookback_snapshot_date,
    cve_id,
    'R-KEV-NEW' AS rule_id,
    'KEV_NEW' AS trigger_type,
    'kev' AS signal,
    60 AS score,
    'false' AS previous_value,
    'true' AS current_value,
    'boolean' AS value_type,
    CAST(signal_snapshot_date AS timestamp) AS evidence_observed_at,
    'CISA KEV' AS source,
    CAST(signal_snapshot_date AS timestamp) AS source_snapshot_at,
    signal_snapshot_date AS source_snapshot_date,
    concat(
        'CISA KEV에 ',
        date_format(signal_snapshot_date, 'yyyy-MM-dd'),
        ' 신규 등재되었습니다.'
    ) AS evidence_summary
FROM {VALIDATION_SIGNAL_VIEW}
WHERE kev_new_in_window
""")

display(spark.sql(f"""
SELECT
    rule_id,
    trigger_type,
    COUNT(*) AS hit_count,
    COUNT(DISTINCT cve_id) AS cve_count,
    MIN(signal_snapshot_date) AS min_signal_snapshot_date,
    MAX(signal_snapshot_date) AS max_signal_snapshot_date
FROM {VALIDATION_RULE_HITS_VIEW}
GROUP BY rule_id, trigger_type
ORDER BY rule_id, trigger_type
"""))

display(spark.sql(f"""
SELECT
    s.epss_lookback_missing_reason AS epss_lookback_missing_reason,
    COUNT(*) AS top5_hit_count,
    COUNT(DISTINCT h.cve_id) AS cve_count
FROM {VALIDATION_RULE_HITS_VIEW} h
JOIN {VALIDATION_SIGNAL_VIEW} s
    ON h.signal_snapshot_date = s.signal_snapshot_date
   AND h.cve_id = s.cve_id
WHERE h.rule_id = 'R-EPSS-TOP5-ENTRY-D30'
GROUP BY s.epss_lookback_missing_reason
ORDER BY s.epss_lookback_missing_reason
"""))


StatementMeta(, be2357d1-6e2d-494d-9071-b0380964338b, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9a7eaa60-bec5-4bb3-a856-cce06342a198)

SynapseWidget(Synapse.DataFrame, 804464b9-3897-4115-8885-ed7b8c9aec2a)

## 4. DS Gold 형태 후보와 근거 생성

A축 과거 후보 결과를 DS 스키마의 Gold 테이블 스키마에 맞게 변환합니다. GHSA와 패키지 grain은 이 기간에서 재현하지 않으므로 관련 컬럼은 `NULL`로 둡니다.

In [10]:
# ==============================================================================
# [Cell 9] Fabric 완벽 검증 슬라이딩 30일 쿨다운 및 뷰 생성
# ==============================================================================
import pandas as pd
from pyspark.sql import functions as F

# 1. 상단 Cell 7 룰 뷰 로드 및 윈도우 컬럼 사전 정의
df_hits = spark.table(VALIDATION_RULE_HITS_VIEW) \
    .withColumn("trigger_family", F.when(F.col("trigger_type") == "KEV_NEW", "KEV").otherwise("EPSS")) \
    .withColumn("trigger_cooldown_window", F.lit(0).cast("int")) \
    .withColumn("candidate_cooldown_window", F.lit(0).cast("int"))

# 2. 슬라이딩 30일 쿨다운 함수 (날짜별 그룹핑)
def apply_sliding_30d_cooldown(pdf: pd.DataFrame) -> pd.DataFrame:
    pdf = pdf.sort_values("signal_snapshot_date")
    selected_rows = []
    last_date = None
    window_seq = 0
    
    for current_date, date_group in pdf.groupby("signal_snapshot_date", sort=False):
        if last_date is None or (current_date - last_date).days >= 30:
            for idx, row in date_group.iterrows():
                row_dict = row.to_dict()
                row_dict["trigger_cooldown_window"] = int(window_seq)
                row_dict["candidate_cooldown_window"] = int(window_seq)
                selected_rows.append(row_dict)
            last_date = current_date
            window_seq += 1
            
    return pd.DataFrame(selected_rows)

# 3. 분산 실행 및 영구 델타 테이블로 안전하게 물리 저장
spark.sql(f"DROP Table IF EXISTS {VALIDATION_FIRST_RULE_HITS_VIEW}")
df_selected_hits = df_hits.groupBy("cve_id", "trigger_family").applyInPandas(
    apply_sliding_30d_cooldown, 
    schema=df_hits.schema
)
df_selected_hits.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(VALIDATION_FIRST_RULE_HITS_VIEW)

# 4. Candidate 뷰 생성
spark.sql(f"""
CREATE OR REPLACE VIEW {VALIDATION_CANDIDATES_VIEW} AS
WITH candidate_events AS (
    SELECT
        cve_id,
        trigger_family,
        candidate_cooldown_window,
        MIN(signal_snapshot_date) AS signal_snapshot_date,
        MIN(lookback_snapshot_date) AS lookback_snapshot_date,
        SUM(score) AS score_total,
        MAX(score) AS risk_change_score,
        collect_set(trigger_type) AS trigger_types,
        MAX(CASE WHEN trigger_type = 'KEV_NEW' THEN 1 ELSE 0 END) AS has_kev_new,
        MAX(CASE WHEN trigger_type = 'EPSS_TOP5_ENTRY' THEN 1 ELSE 0 END) AS has_epss_top5,
        MAX(CASE WHEN trigger_type = 'EPSS_RISE' THEN 1 ELSE 0 END) AS has_epss_rise
    FROM {VALIDATION_FIRST_RULE_HITS_VIEW}
    GROUP BY
        cve_id,
        trigger_family,
        candidate_cooldown_window
)
SELECT
    '{BATCH_ID}' AS batch_id,
    concat('cand-', substr(sha2(concat_ws('|', '{BATCH_ID}', cve_id, trigger_family, CAST(signal_snapshot_date AS string)), 256), 1, 16)) AS candidate_id,
    signal_snapshot_date,
    lookback_snapshot_date,
    CAST(NULL AS string) AS ghsa_id,
    cve_id,
    CAST(NULL AS string) AS package_ecosystem,
    CAST(NULL AS string) AS package_name,
    CAST(NULL AS string) AS affected_range,
    CASE
        WHEN has_kev_new = 1 THEN 'P0'
        WHEN score_total >= 45 THEN 'P1'
        WHEN score_total >= 30 THEN 'P2'
        ELSE 'P3'
    END AS priority,
    score_total,
    risk_change_score,
    0 AS current_severity_score,
    0 AS actionability_change_score,
    trigger_types,
    CASE
        WHEN has_kev_new = 1 THEN 'KEV_NEW'
        WHEN has_epss_top5 = 1 THEN 'EPSS_TOP5_ENTRY'
        WHEN has_epss_rise = 1 THEN 'EPSS_RISE'
    END AS primary_trigger_type,
    CASE
        WHEN has_kev_new = 1 THEN 'CISA KEV에 신규 등재되었습니다.'
        WHEN has_epss_top5 = 1 THEN 'EPSS 위험 순위가 30일 전 대비 상위 5% 구간에 진입했습니다.'
        WHEN has_epss_rise = 1 THEN 'EPSS가 30일 전 대비 급상승했습니다.'
    END AS reason_summary,
    CAST(signal_snapshot_date AS timestamp) AS observed_at,
    timestamp('{GENERATED_AT_UTC}') AS selected_at,
    trigger_family,
    candidate_cooldown_window
FROM candidate_events
""")

# 5. Evidence 뷰 생성
spark.sql(f"""
CREATE OR REPLACE VIEW {VALIDATION_EVIDENCE_VIEW} AS
SELECT
    c.batch_id,
    c.candidate_id,
    concat('ev_', substr(sha2(concat_ws('|', c.candidate_id, h.rule_id, h.signal, CAST(h.signal_snapshot_date AS string), COALESCE(h.previous_value, ''), COALESCE(h.current_value, '')), 256), 1, 16)) AS evidence_id,
    h.rule_id,
    h.signal,
    h.previous_value,
    h.current_value,
    h.value_type,
    h.evidence_observed_at,
    h.source,
    h.source_snapshot_at,
    h.source_snapshot_date,
    h.evidence_summary
FROM {VALIDATION_FIRST_RULE_HITS_VIEW} h
JOIN {VALIDATION_CANDIDATES_VIEW} c
    ON h.cve_id = c.cve_id
   AND h.trigger_family = c.trigger_family
   AND h.candidate_cooldown_window = c.candidate_cooldown_window
""")

# 6. 확인용 집계 출력
display(spark.sql(f"""
SELECT
    signal_snapshot_date,
    primary_trigger_type,
    COUNT(*) AS candidate_count
FROM {VALIDATION_CANDIDATES_VIEW}
GROUP BY signal_snapshot_date, primary_trigger_type
ORDER BY signal_snapshot_date, primary_trigger_type
"""))

StatementMeta(, be2357d1-6e2d-494d-9071-b0380964338b, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a3be997e-d2da-48a9-a3be-395a5d970405)

## 5. 적재 전 규모와 중복 확인

DS Gold 테이블에 append하기 전에 일자별 후보 수, 반복 CVE, 룰별 최초 hit 기준 후보 수를 확인합니다. 결과가 과도하면 이 셀에서 기준을 확정한 뒤 적재 방식을 조정합니다.

In [11]:
display(spark.sql(f"""
WITH repeated_cves AS (
    SELECT
        cve_id,
        COUNT(*) AS candidate_count
    FROM {VALIDATION_CANDIDATES_VIEW}
    GROUP BY cve_id
    HAVING COUNT(*) > 1
)
SELECT
    c.signal_snapshot_date,
    COUNT(*) AS candidate_count,
    COUNT(DISTINCT c.cve_id) AS distinct_cve_count,
    SUM(CASE WHEN r.cve_id IS NOT NULL THEN 1 ELSE 0 END) AS repeated_cve_candidate_count,
    COUNT(DISTINCT CASE WHEN r.cve_id IS NOT NULL THEN c.cve_id END) AS repeated_distinct_cve_count,
    SUM(CASE WHEN c.primary_trigger_type = 'KEV_NEW' THEN 1 ELSE 0 END) AS kev_new_candidate_count,
    SUM(CASE WHEN c.primary_trigger_type = 'EPSS_TOP5_ENTRY' THEN 1 ELSE 0 END) AS epss_top5_entry_candidate_count,
    SUM(CASE WHEN c.primary_trigger_type = 'EPSS_RISE' THEN 1 ELSE 0 END) AS epss_rise_candidate_count
FROM {VALIDATION_CANDIDATES_VIEW} c
LEFT JOIN repeated_cves r
    ON c.cve_id = r.cve_id
GROUP BY c.signal_snapshot_date
ORDER BY c.signal_snapshot_date
"""))

display(spark.sql(f"""
WITH cve_counts AS (
    SELECT
        cve_id,
        COUNT(*) AS candidate_row_count,
        COUNT(DISTINCT primary_trigger_type) AS primary_trigger_type_count,
        COUNT(DISTINCT trigger_family) AS trigger_family_count
    FROM {VALIDATION_CANDIDATES_VIEW}
    GROUP BY cve_id
)
SELECT
    SUM(candidate_row_count) AS candidate_row_count,
    COUNT(*) AS distinct_cve_count,
    SUM(CASE WHEN candidate_row_count > 1 THEN 1 ELSE 0 END) AS repeated_distinct_cve_count,
    SUM(CASE WHEN candidate_row_count > 1 THEN candidate_row_count ELSE 0 END) AS repeated_cve_candidate_row_count,
    SUM(CASE WHEN candidate_row_count > 1 THEN candidate_row_count - 1 ELSE 0 END) AS extra_repeated_candidate_count,
    MAX(candidate_row_count) AS max_candidate_count_per_cve
FROM cve_counts
"""))

display(spark.sql(f"""
SELECT
    c.cve_id,
    COUNT(*) AS candidate_count,
    MIN(c.signal_snapshot_date) AS first_candidate_date,
    MAX(c.signal_snapshot_date) AS last_candidate_date,
    collect_set(c.primary_trigger_type) AS primary_trigger_types,
    collect_set(c.trigger_family) AS trigger_families,
    collect_set(c.candidate_cooldown_window) AS candidate_cooldown_windows
FROM {VALIDATION_CANDIDATES_VIEW} c
GROUP BY c.cve_id
HAVING COUNT(*) > 1
ORDER BY candidate_count DESC, first_candidate_date, cve_id
LIMIT 100
"""))

display(spark.sql(f"""
WITH repeated_cves AS (
    SELECT cve_id
    FROM {VALIDATION_CANDIDATES_VIEW}
    GROUP BY cve_id
    HAVING COUNT(*) > 1
)
SELECT
    c.cve_id,
    c.candidate_id,
    c.signal_snapshot_date AS candidate_signal_snapshot_date,
    c.primary_trigger_type,
    c.trigger_types,
    c.priority,
    c.score_total,
    h.signal_snapshot_date AS evidence_signal_snapshot_date,
    h.lookback_snapshot_date,
    h.rule_id,
    h.trigger_type,
    h.signal,
    h.previous_value,
    h.current_value,
    h.value_type,
    h.evidence_summary
FROM {VALIDATION_CANDIDATES_VIEW} c
JOIN {VALIDATION_FIRST_RULE_HITS_VIEW} h
    ON c.cve_id = h.cve_id
   AND c.trigger_family = h.trigger_family
   AND c.candidate_cooldown_window = h.candidate_cooldown_window
JOIN repeated_cves r
    ON c.cve_id = r.cve_id
ORDER BY c.cve_id, c.signal_snapshot_date, h.signal_snapshot_date, h.rule_id
"""))

display(spark.sql(f"""
SELECT
    rule_id,
    trigger_type,
    COUNT(*) AS selected_hit_count,
    COUNT(DISTINCT cve_id) AS distinct_cve_count,
    MIN(signal_snapshot_date) AS first_signal_snapshot_date,
    MAX(signal_snapshot_date) AS last_signal_snapshot_date
FROM {VALIDATION_FIRST_RULE_HITS_VIEW}
GROUP BY rule_id, trigger_type
ORDER BY rule_id, trigger_type
"""))

display(spark.sql(f"""
SELECT
    signal_snapshot_date,
    rule_id,
    trigger_type,
    COUNT(*) AS selected_hit_count
FROM {VALIDATION_FIRST_RULE_HITS_VIEW}
GROUP BY signal_snapshot_date, rule_id, trigger_type
ORDER BY signal_snapshot_date, rule_id, trigger_type
"""))


StatementMeta(, be2357d1-6e2d-494d-9071-b0380964338b, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7d0d2999-1573-4c42-aae7-c7d678bf4d97)

SynapseWidget(Synapse.DataFrame, c8689be9-f557-4e41-b6d1-bc60c267fbc5)

SynapseWidget(Synapse.DataFrame, 33ad8c6d-e64b-413a-8bba-aa027cd934c8)

SynapseWidget(Synapse.DataFrame, 4e776a93-9586-45bb-95ae-96b49b11629a)

SynapseWidget(Synapse.DataFrame, 0736bbb7-f84f-467e-a4ea-e0d108b7b7fe)

SynapseWidget(Synapse.DataFrame, 23d4ebf1-5483-4df0-8e25-4f81070c710b)

## 6. DS Gold 테이블 적재

`WRITE_OUTPUT_TABLES`가 `True`일 때만 DS 스키마의 Gold 형태 테이블에 append합니다.

In [12]:
candidate_count = spark.sql(f"""
SELECT COUNT(*) AS cnt
FROM {VALIDATION_CANDIDATES_VIEW}
""").collect()[0]["cnt"]

evidence_count = spark.sql(f"""
SELECT COUNT(*) AS cnt
FROM {VALIDATION_EVIDENCE_VIEW}
""").collect()[0]["cnt"]

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW historical_batch AS
SELECT
    '{BATCH_ID}' AS batch_id,
    '{RUN_ID}' AS run_id,
    'completed' AS status,
    DATE('{END_SNAPSHOT_DATE}') AS as_of,
    timestamp('{GENERATED_AT_UTC}') AS generated_at,
    '{POLICY_VERSION}' AS policy_version,
    '{SCHEMA_VERSION}' AS schema_version,
    {candidate_count} AS candidate_count
""")

if WRITE_OUTPUT_TABLES:
    spark.sql(f"""
    INSERT INTO {DS_CANDIDATE_TABLE} (
        batch_id,
        candidate_id,
        signal_snapshot_date,
        lookback_snapshot_date,
        ghsa_id,
        cve_id,
        package_ecosystem,
        package_name,
        affected_range,
        priority,
        score_total,
        risk_change_score,
        current_severity_score,
        actionability_change_score,
        trigger_types,
        primary_trigger_type,
        reason_summary,
        observed_at,
        selected_at
    )
    SELECT
        batch_id,
        candidate_id,
        signal_snapshot_date,
        lookback_snapshot_date,
        ghsa_id,
        cve_id,
        package_ecosystem,
        package_name,
        affected_range,
        priority,
        score_total,
        risk_change_score,
        current_severity_score,
        actionability_change_score,
        trigger_types,
        primary_trigger_type,
        reason_summary,
        observed_at,
        selected_at
    FROM {VALIDATION_CANDIDATES_VIEW}
    """)

    spark.sql(f"""
    INSERT INTO {DS_EVIDENCE_TABLE} (
        batch_id,
        candidate_id,
        evidence_id,
        rule_id,
        signal,
        previous_value,
        current_value,
        value_type,
        evidence_observed_at,
        source,
        source_snapshot_at,
        source_snapshot_date,
        evidence_summary
    )
    SELECT
        batch_id,
        candidate_id,
        evidence_id,
        rule_id,
        signal,
        previous_value,
        current_value,
        value_type,
        evidence_observed_at,
        source,
        source_snapshot_at,
        source_snapshot_date,
        evidence_summary
    FROM {VALIDATION_EVIDENCE_VIEW}
    """)

    spark.sql(f"""
    INSERT INTO {DS_BATCH_TABLE} (
        batch_id,
        run_id,
        status,
        as_of,
        generated_at,
        policy_version,
        schema_version,
        candidate_count
    )
    SELECT
        batch_id,
        run_id,
        status,
        as_of,
        generated_at,
        policy_version,
        schema_version,
        candidate_count
    FROM historical_batch
    """)

print({
    "write_output_tables": WRITE_OUTPUT_TABLES,
    "batch_id": BATCH_ID,
    "candidate_count": candidate_count,
    "evidence_count": evidence_count,
})


StatementMeta(, be2357d1-6e2d-494d-9071-b0380964338b, 14, Finished, Available, Finished, False)

{'write_output_tables': True, 'batch_id': 'historical-20260827T043617Z', 'candidate_count': 1667, 'evidence_count': 2138}


## 7. 적재 결과 확인

이번 과거 후보 생성 배치 기준으로 DS Gold 테이블에 들어간 결과를 확인합니다.

In [13]:
display(spark.sql(f"""
SELECT *
FROM {DS_BATCH_TABLE}
WHERE batch_id = '{BATCH_ID}'
"""))

display(spark.sql(f"""
SELECT
    signal_snapshot_date,
    primary_trigger_type,
    COUNT(*) AS candidate_count
FROM {DS_CANDIDATE_TABLE}
WHERE batch_id = '{BATCH_ID}'
GROUP BY signal_snapshot_date, primary_trigger_type
ORDER BY signal_snapshot_date, primary_trigger_type
"""))

display(spark.sql(f"""
SELECT
    rule_id,
    COUNT(*) AS evidence_count
FROM {DS_EVIDENCE_TABLE}
WHERE batch_id = '{BATCH_ID}'
GROUP BY rule_id
ORDER BY rule_id
"""))

StatementMeta(, be2357d1-6e2d-494d-9071-b0380964338b, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0581aeae-7090-4bc4-afa6-465b401f1722)

SynapseWidget(Synapse.DataFrame, e075c3fd-e63e-4f63-82c8-ce7075aca247)

SynapseWidget(Synapse.DataFrame, a075ce42-6ac5-4bdf-b4ea-453bd29ceceb)

In [15]:
LAKEHOUSE = "`Watchback-workspace`.watchback_lakehouse_dev_v2"
DS_CANDIDATE_TABLE = f"{LAKEHOUSE}.DS.gold_rereview_candidate"

# 1. 전체 요약 (고유 CVE 수, 일자 등)
print("📊 [1. 전체 요약 지표]")
display(spark.sql(f"""
SELECT
    COUNT(*) AS total_candidates,
    COUNT(DISTINCT cve_id) AS distinct_cves,
    MIN(signal_snapshot_date) AS first_date,
    MAX(signal_snapshot_date) AS last_date,
    COUNT(DISTINCT signal_snapshot_date) AS snapshot_days
FROM {DS_CANDIDATE_TABLE}
"""))

# 2. 트리거 타입별 후보 수 및 고유 CVE 수
print("\n📊 [2. 트리거 타입별 고유 CVE 수]")
display(spark.sql(f"""
SELECT
    primary_trigger_type,
    COUNT(*) AS total_candidates,
    COUNT(DISTINCT cve_id) AS distinct_cves,
    COUNT(*) - COUNT(DISTINCT cve_id) AS multi_candidate_cves
FROM {DS_CANDIDATE_TABLE}
GROUP BY primary_trigger_type
ORDER BY total_candidates DESC
"""))

# 3. 🎯 CVE-2018-1320 중복 해결 확인 (기존 3건 -> 2건으로 정상화되었는지)
print("\n📋 [3. CVE-2018-1320 확인]")
display(spark.sql(f"""
SELECT
    signal_snapshot_date,
    cve_id,
    primary_trigger_type,
    priority,
    score_total,
    candidate_id
FROM {DS_CANDIDATE_TABLE}
WHERE cve_id = 'CVE-2018-1320'
ORDER BY signal_snapshot_date
"""))

StatementMeta(, be2357d1-6e2d-494d-9071-b0380964338b, 17, Finished, Available, Finished, False)

📊 [1. 전체 요약 지표]


SynapseWidget(Synapse.DataFrame, 2e05197a-910b-4b8f-9557-822a9a094c5e)


📊 [2. 트리거 타입별 고유 CVE 수]


SynapseWidget(Synapse.DataFrame, ddc66321-d41d-4d24-8a51-356b3f2a2255)


📋 [3. CVE-2018-1320 확인]


SynapseWidget(Synapse.DataFrame, 29c3f4d0-e260-48ef-a700-7b0d53dacb04)

In [16]:
display(spark.sql(f"""
SELECT
    signal_snapshot_date,
    cve_id,
    primary_trigger_type,
    priority,
    score_total,
    candidate_id
FROM {DS_CANDIDATE_TABLE}
WHERE signal_snapshot_date = '2026-06-15'
ORDER BY signal_snapshot_date
"""))

StatementMeta(, be2357d1-6e2d-494d-9071-b0380964338b, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 37243794-840e-4aac-9055-95690d82a846)